# Stock Price Prediction using Random Forest

Predict AAPL closing prices using historical market data downloaded with `yfinance` and a Random Forest Regressor.


In [ ]:
# Install once if needed
# %pip install yfinance pandas numpy matplotlib scikit-learn


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

stock = "AAPL"
data = yf.download(stock, start="2015-01-01", end="2024-01-01", auto_adjust=True)

# Handle yfinance MultiIndex output when present
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data.reset_index()
data.columns = [str(c).strip().lower() for c in data.columns]
data.to_csv("data/stock_data.csv", index=False)
data.head()


In [ ]:
# Data preprocessing
required = ["open", "high", "low", "close", "volume"]
data[required] = data[required].apply(pd.to_numeric, errors="coerce")
data = data.dropna().copy()

# Feature engineering
data["ma_10"] = data["close"].rolling(10).mean()
data["ma_50"] = data["close"].rolling(50).mean()
data["target"] = data["close"].shift(-1)
data = data.dropna().copy()

data.head()


In [ ]:
# Time-ordered train/test split
feature_cols = ["open", "high", "low", "volume", "ma_10", "ma_50"]
X = data[feature_cols]
y = data["target"]

split = int(len(data) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
predictions = model.predict(X_test)


In [ ]:
# Model evaluation
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2 Score: {r2:.4f}")


In [ ]:
# Actual vs predicted prices
plt.figure(figsize=(12, 5))
plt.plot(y_test.index, y_test.values, label="Actual")
plt.plot(y_test.index, predictions, label="Predicted", linestyle="--")
plt.title("AAPL Actual vs Predicted Closing Price")
plt.xlabel("Test Sample")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()
